# HRAI API demo notebook

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

!git lfs pull

In [1]:
import os, json, random, textwrap, requests
from pathlib import Path

from config import conf
from load import get_encoder
from query import query_type, extract_from_resume
from suggestions import get_expanded_skills, get_domain_reports
from pos_extraction import text_to_ngrams

from dotenv import load_dotenv
import faiss

BASE_URL = 'http://127.0.0.1:8000'

## konfigurace modelu a metadat pro lokální volání
\*pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN v .env

In [2]:
load_dotenv()
db = faiss.read_index(os.path.join(conf.db_dir, f"all.index"))
with open(os.path.join(conf.data_dir, f"key_to_ent.json"),'r') as f:
    metadata = json.loads(f.read())
model = get_encoder()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [42]:
def print_skill(s):
    print(f'{s['label']}')
    if s['description']: print(f'informace: {s['description']}\n')

def print_occupation(occ):
    print(
        f'{occ['label']}\n'
        f'skóre: {occ['cosine_score']}'
    )
    if occ['description']: print(f'info: {occ['description']}\n')

def print_missing_skills(skills):
    for s in skills:
        if 'missing_skills' in s and len(s['missing_skills']) == 0: continue
        if 'occupation' in s: print_occupation(s['occupation'])
        if 'missing_skills' not in s: continue
        essential = [s for s in s['missing_skills']if s['relation_type'] == 'essential']
        optional = [s for s in s['missing_skills']if s['relation_type'] == 'optional']

        print('Základní dovednosti/znalosti pro výkon pozice:')
        for e in essential: print_skill(e)

        print('\nDalší užitečné dovednosti:')
        for o in optional: print_skill(o)
        print('-'*100)

def print_domain_info(d):
    for occ in d['occupations']:
        print_occupation(occ['occupation'])
        if 'missing_skills' in occ:
            print('')
            print_missing_skills(occ['missing_skills'])
    print('-'*100+'\n')

## převedení textu na ngramy

In [3]:
resumes = json.loads(Path(os.path.join('testfiles', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)
print(textwrap.shorten(text, width=2000))

Profil výkonného ředitele Zajistit roli manažera korporátní komunikace ve snaze využívat výjimečné verbální a písemné komunikační techniky k implementaci a prosazování strategických cílů organizace Výjimečné písemné a ústní komunikační dovednosti. Zkušenosti s implementací změn v rámci celé instituce s cílem podpořit a povzbudit inkluzi, respekt a důstojnost pro všechny složky. Vysoce kvalifikovaný ve vytváření chutných obchodních případů o výhodách spojených s rozmanitostí a inkluzí. Prokázaná schopnost proaktivně a pečlivě spolupracovat se zúčastněnými stranami a zároveň podporovat cíle organizace v oblasti rozmanitosti a inkluze. Dynamické interpersonální, analytické, organizační schopnosti. : Vedení, Budování vztahů, Strategické obchodní zaměření, Sebezdokonalování, Týmová práce, Rozhodování a úsudek, Přizpůsobivost, Inkluzivita, Agilita, Kvalita, Odpovědnost, Zaměření na zákazníka (interní i externí), Pracovní morálka, Vynalézavost, Komunikace (ústní i písemná), Kritické myšlení, 

In [4]:
ngrams = text_to_ngrams(text)
print('\n'.join(ngrams[:20])+'\n[...]')

vzdělání
prezidenta george
úsudek
uznávaného programu
obhájení
otázkách
klíčový člen
záležitostech
konfliktů
manažerů a zaměstnanců
kalamazoo absolvent
zvládla rozsáhlé problémy
mother of
state university město
působil jako prezident
komunikace lidské zdroje
implementaci a prosazování
zaměstnanci vlastní motivace
rámci celé instituce
cílem
[...]


## API endpointy: /resume/skills a /resume/domains (PDF, DOCX, ODT)

In [43]:
def get_skills(filename: str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/octet-stream')}
    res = requests.post(f"{BASE_URL}/resume/skills", files=files)

    print(f'file: {filename}\n'
          f'status: {res.status_code}\n'
    )
    print_missing_skills(res.json())


def get_domains(filename:str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/octet-stream')}
    res = requests.post(f"{BASE_URL}/resume/domains", files=files)

    print(f'file: {filename}\n'
          f'status: {res.status_code}\n'
    )
    domains = set([d['domain'] for d in res.json()])

    for d in res.json():
        print(f'oblast: {d['domain']}\n'
          f'počet zaměstnání se shodou: {d['occ_count']}\n'
          )
        if 'occupations' not in d: continue
        if len(d['occupations']) <1: continue
        print_domain_info(d)
    return

get_domains('resume.pdf')

file: resume.pdf
status: 200

oblast: Informační a komunikační technologie
počet zaměstnání se shodou: 17

Programátoři počítačových aplikací
skóre: 0.7477958798408508

Vývojáři softwaru
skóre: 0.6735450625419617

Programátoři počítačových aplikací
skóre: 0.846380352973938

Vývojáři softwaru
skóre: 0.8167893886566162

softwarový vývojář/softwarová vývojářka / programátor softwaru
skóre: 0.8635849952697754
info: Vývojáři softwaru implementují nebo programují všechny typy softwarových systémů založených na specifikacích a návrzích s použitím programovacích jazyků, nástrojů a platforem.


Programátoři počítačových aplikací
skóre: 0.682443380355835

Programátoři počítačových aplikací
skóre: 0.7571786046028137

softwarový vývojář/softwarová vývojářka / programátor softwaru
skóre: 0.8013341426849365
info: Vývojáři softwaru implementují nebo programují všechny typy softwarových systémů založených na specifikacích a návrzích s použitím programovacích jazyků, nástrojů a platforem.


Vývojáři so

In [26]:
get_skills('/resume/skills', 'Životopis.pdf')

endpoint: /resume/skills
file: Životopis.pdf
status: 200

kočí
skóre: 0.7223371267318726
info: Kočí vozí cestující v koňských povozech. Zajišťují bezpečnost cestujících a péči o koně.

Základní dovednosti/znalosti pro výkon pozice:
právní předpisy týkající se silničního provozu
informace: Znát právní předpisy o dopravním provozu a pravidla silničního provozu.

chování zvířat
informace: Přirozené vzorce chování zvířat, tj. způsob, jakým lze normální a neobvyklé chování vyjádřit na základě živočišného druhu, životního prostředí, interakce člověka a zvířat a povolání.

soustředit se na potřeby cestujících
informace: Bezpečně a včas přepravovat cestující do jejich místa určení. Poskytovat vhodné zákaznické služby; informovat cestující v případě neočekávaných situací nebo jiných mimořádných událostí.

komunikovat se zákazníky
informace: Odpovídat zákazníkům a komunikovat s nimi co nejúčinnějším a nejvhodnějším způsobem, aby jim byl umožněn přístup k požadovaným výrobkům nebo službám nebo k 

In [ ]:
post_file('/resume/domains', 'resume.pdf')

## API endpointy: /text/skills a /text/domains

In [ ]:
text_req = {
    'occupations': ['software developer', 'data analyst'],
    'skills': ['python', 'strojové učení', 'sql databáze']
}
skills_resp = requests.post(f"{BASE_URL}/text/skills", json=text_req)
print({'endpoint': '/text/skills', 'status': skills_resp.status_code, 'json': skills_resp.json()[0]})

domains_resp = requests.post(f"{BASE_URL}/text/domains", json=text_req)
print({'endpoint': '/text/domains', 'status': domains_resp.status_code, 'json': domains_resp.json()[0]})

## API endpoint: /query (každý typ dotazu)

In [ ]:
query_cases = [
    {'query': 'finančník', 'query_type': 'occupation'},
    {'query': 'napomínání lidí', 'query_type': 'skill', 'min_set_score':0.5},
    {'query': 'řemesla', 'query_type': 'isco_group'},
    {'query': 'správa sítí', 'query_type': 'skill_group'},
]
for case in query_cases:
    resp = requests.post(f"{BASE_URL}/query", json=case)
    print({'endpoint': '/query', 'request': case, 'status': resp.status_code, 'json': resp.json()})

## Přímé volání query_type pro všechny ent_type

In [ ]:
entry = ['rybář']
ent_type = 'occupation'

results = query_type(db=db, metadata=metadata, model=model, ents=entry, ent_type=ent_type, min_score=0.5)
print(f"{entry} ({ent_type})")
print(results)

## Pipeline: extract_from_resume -> get_expanded_skills -> get_domain_reports

In [ ]:
random_resume = random.choice(resumes)
print(textwrap.shorten(f'sample text: {random_resume}', width=600, placeholder="..."))

extracted_entities = extract_from_resume(db, metadata, model, random_resume)
print(textwrap.shorten(f'extracted entities: {extracted_entities}', width=600, placeholder="..."))

skill_suggestions = get_expanded_skills(metadata, extracted_entities)
print(textwrap.shorten(f'skill suggestions: {skill_suggestions}', width=600, placeholder="..."))

domain_reports = get_domain_reports(skill_suggestions)
print(textwrap.shorten(f'domain stats: {domain_reports}', width=600, placeholder="..."))